In [ ]:
import kagglehub
kagglehub.login()

In [ ]:
kagglehub.competition_download(
    'kmu-rec-sys-26-rating-prediction',
    output_dir = "dataset"
)

In [ ]:
!head dataset/train_small.csv

baseline.py

In [ ]:
import csv

# 1. train_small.csv에서 전체 평균 계산
total, count = 0.0, 0
with open('dataset/train_small.csv', newline='') as f:
    for row in csv.DictReader(f):
        total += float(row['rating'])
        count += 1

mean_rating = total / count

# 2. test_small.csv를 읽어 모든 예측값을 평균으로 채워 저장
with open('dataset/test_small.csv', newline='') as fin, \
     open('baseline_solution.csv', 'w', newline='') as fout:
    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=['id', 'rating'])
    writer.writeheader()
    for row in reader:
        writer.writerow({'id': f"{row['userId']}_{row['movieId']}", 'rating': mean_rating})


In [ ]:
!head baseline_solution.csv

##Bias 모델 - 데이터 준비

### - userId와 itemId를 리매핑
sparse -> dense

### - users, items, ratings를 numpy array 형태로 변환

In [ ]:
import numpy as np

userIds, itemIds = {}, {}
n_users, n_items, n_ratings = 0,0,0
users, items, ratings = [], [], []

with open('dataset/train_small.csv') as f:
  for row in csv.DictReader(f):
    if row['userId'] not in userIds:
      userIds[row['userId']] = n_users
      n_users += 1
    if row['movieId'] not in itemIds:
      itemIds[row['movieId']] = n_items
      n_items += 1

    n_ratings += 1

    uid = userIds[row['userId']]
    iid = itemIds[row['movieId']]
    rating = float(row['rating'])

    users.append(uid)
    items.append(iid)
    ratings.append(rating)

  users = np.array(users)
  items = np.array(items)
  ratings = np.array(ratings)


##Bias 모델 - Gradient Descent

In [ ]:
import numpy as np

mean = ratings.mean()
user_bias = np.zeros(n_users)
item_bias = np.zeros(n_items)

lr = 0.0001
lmd = 0.001

for epoch in range(1000):
  h = mean + user_bias[users] + item_bias[items]
  diff = h - ratings
  print(f"epoch: {epoch}, mse: {(diff**2).mean()}")

  user_grad = np.bincount(users, weights=diff) + lmd*user_bias
  item_grad = np.bincount(items, weights=diff) + lmd*item_bias

  user_bias -= lr*user_grad
  item_bias -= lr*item_grad

In [ ]:
with open('dataset/test_small.csv') as fin, \
     open('bias_solution.csv','w', newline='') as fout:

    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=['id','rating'])
    writer.writeheader()

    for row in reader:
        uid = userIds[row['userId']]
        iid = itemIds[row['movieId']]

        rating_pred = mean + user_bias[uid] + item_bias[iid]
        rating_pred = max(0, min(5, rating_pred))

        if hasattr(rating_pred, 'item'):
            rating_pred = rating_pred.item()

        writer.writerow({
            'id': f"{row['userId']}_{row['movieId']}",
            'rating': rating_pred
        })

In [ ]:
!head bias_solution.csv

## Bias 모델 - Gradient Descent (Pytorch)

In [ ]:
import torch

lr = 0.013
lmd = 0.0001

users = torch.LongTensor(users)
items = torch.LongTensor(items)
ratings = torch.FloatTensor(ratings)

user_bias = torch.zeros(n_users, requires_grad=True)
item_bias = torch.zeros(n_items, requires_grad=True)

optim = torch.optim.Adam(params=[user_bias, item_bias], lr=lr)
mse = torch.nn.MSELoss()

for epoch in range(1000):
  h = mean + user_bias[users] + item_bias[items]
  cost = mse(h, ratings)
  cost += lmd * (user_bias**2).sum()
  cost += lmd * (item_bias**2).sum()

  optim.zero_grad()
  cost.backward()
  optim.step()

  print(f"epoch:{epoch}, mse:{cost.item()}")

##Bias 모델 - Alterating Least Squares

In [ ]:
lmd = 10

mean = ratings.mean()
user_bias = np.zeros(n_users)
item_bias = np.zeros(n_items)

for epoch in range(50):
  h = mean + user_bias[users] + item_bias[items]
  mse = ((h-ratings)**2).mean()
  print(f"epoch:{epoch}, mse:{mse}")

  tmp = rating - (mean+item_bias[items])
  user_bias = np.bincount(users, weights=tmp) / (np.bincount(users)+lmd)
  tmp = ratings - (mean+user_bias[users])
  item_bias = np.bincount(items,weights=tmp) / (np.bincount(items)+lmd)